<a href="https://colab.research.google.com/github/Markinhoow/Atividade-Parcial---IA/blob/main/Lista_N2_Inteligencia_Artificial_Marco_Antonio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lista de Exercícios N2 — Grafos e Algoritmos de Busca

**Disciplina:** Inteligência Artificial  
**Ambiente:** Google Colab  
**Entrega:** Repositório individual no GitHub  
**Nome:** Marco Antonio Carvalho Freitas de Souza  
**Matrícula:** 2249178  


## Parte A — Dijkstra (Caminho Mínimo em Grafos Ponderados Positivos)

Imagine um sistema de navegação para ambulâncias. Cada interseção é um nó, cada rua é uma aresta ponderada pelo tempo médio de deslocamento. Nosso objetivo é encontrar a rota mais rápida do hospital até o local de atendimento.


In [1]:
import heapq
import matplotlib.pyplot as plt
from dataclasses import dataclass

inf = float('inf')

def dijkstra(grafo, inicio):
    dist = {no: inf for no in grafo}
    parent = {no: None for no in grafo}
    dist[inicio] = 0
    fila_prioridade = [(0, inicio)]

    while fila_prioridade:
        dist_atual, no_atual = heapq.heappop(fila_prioridade)
        if dist_atual > dist[no_atual]:
            continue
        for vizinho, peso in grafo[no_atual]:
            nova_dist = dist_atual + peso
            if nova_dist < dist[vizinho]:
                dist[vizinho] = nova_dist
                parent[vizinho] = no_atual
                heapq.heappush(fila_prioridade, (nova_dist, vizinho))
    return dist, parent

def reconstruir_caminho(parent, alvo):
    caminho = []
    no = alvo
    while no is not None:
        caminho.append(no)
        no = parent[no]
    return caminho[::-1]

grafo_cidade = {
    'H': [('A', 5), ('B', 10)],
    'A': [('H', 5), ('C', 3), ('D', 11)],
    'B': [('H', 10), ('D', 5)],
    'C': [('A', 3), ('E', 8)],
    'D': [('A', 11), ('B', 5), ('E', 2)],
    'E': [('C', 8), ('D', 2)]
}

inicio = 'H'
fim = 'E'
distancias, predecessores = dijkstra(grafo_cidade, inicio)
caminho = reconstruir_caminho(predecessores, fim)

print("--- Parte A: Teste do Dijkstra ---")
print(f"Menor distância de '{inicio}' para '{fim}': {distancias[fim]}")
print(f"Caminho: {' -> '.join(caminho)}")

--- Parte A: Teste do Dijkstra ---
Menor distância de 'H' para 'E': 16
Caminho: H -> A -> C -> E


### Questões Discursivas — Dijkstra

**1. Por que Dijkstra exige arestas não negativas?**  
Porque o algoritmo é guloso: ele assume que, ao visitar um nó, o caminho encontrado até ele é definitivo. Se existirem arestas negativas, um caminho posterior poderia reduzir a distância de um nó já visitado, quebrando a corretude.

**2. Qual é a complexidade do algoritmo Dijkstra utilizando lista de adjacência e heapq?**  
A complexidade é **O(E log V)**, onde **V** é o número de vértices e **E** é o número de arestas. Cada aresta pode gerar uma atualização de custo na fila de prioridade, operação que custa **O(log V)**.


## Parte B — A* (Busca Informada com Heurística Admissível)

Um robô precisa atravessar um labirinto 2D evitando obstáculos. Diferente do Dijkstra, o A* usa heurística para priorizar caminhos promissores.


In [2]:
import numpy as np

def gerar_grade(rows, cols, obstacle_ratio):
    grid = np.zeros((rows, cols), dtype=int)
    num_obstacles = int(rows * cols * obstacle_ratio)
    indices_obs = np.random.choice(rows * cols, num_obstacles, replace=False)
    coords_obs = np.unravel_index(indices_obs, (rows, cols))
    grid[coords_obs] = 1
    return grid

def heuristic(a, b):
    (x1, y1), (x2, y2) = a, b
    return abs(x1 - x2) + abs(y1 - y2)

def a_star(grid, start, goal, funcao_heuristica):
    rows, cols = grid.shape
    vizinhos = [(0, 1), (0, -1), (1, 0), (-1, 0)]
    g_score = {(r, c): float('inf') for r in range(rows) for c in range(cols)}
    g_score[start] = 0
    f_score = {(r, c): float('inf') for r in range(rows) for c in range(cols)}
    f_score[start] = funcao_heuristica(start, goal)
    fila_aberta = [(f_score[start], start)]
    parent = {}

    while fila_aberta:
        f_atual, no_atual = heapq.heappop(fila_aberta)
        if no_atual == goal:
            path = []
            while no_atual in parent:
                path.append(no_atual)
                no_atual = parent[no_atual]
            path.append(start)
            return path[::-1]
        if f_atual > f_score[no_atual]:
            continue
        r, c = no_atual
        for dr, dc in vizinhos:
            vizinho = (r + dr, c + dc)
            if 0 <= vizinho[0] < rows and 0 <= vizinho[1] < cols:
                if grid[vizinho[0]][vizinho[1]] == 1:
                    continue
                tentative_g = g_score[no_atual] + 1
                if tentative_g < g_score[vizinho]:
                    parent[vizinho] = no_atual
                    g_score[vizinho] = tentative_g
                    f_score[vizinho] = tentative_g + funcao_heuristica(vizinho, goal)
                    heapq.heappush(fila_aberta, (f_score[vizinho], vizinho))
    return None

GRID_SIZE = 20
OBSTACLE_RATIO = 0.15
grid = gerar_grade(GRID_SIZE, GRID_SIZE, OBSTACLE_RATIO)
start_pos, goal_pos = (0, 0), (GRID_SIZE - 1, GRID_SIZE - 1)
grid[start_pos] = 0
grid[goal_pos] = 0

path = a_star(grid, start_pos, goal_pos, heuristic)
print("\n--- Parte B: Teste do A* ---")
if path:
    print(f"Caminho encontrado! Comprimento: {len(path)-1} passos")
else:
    print("Caminho não encontrado.")


--- Parte B: Teste do A* ---
Caminho encontrado! Comprimento: 38 passos


### Questões Discursivas — A*

**1. A * vs Dijkstra: qual expande menos nós?**  
O A* expande menos nós porque utiliza uma heurística que orienta a busca em direção ao objetivo, priorizando caminhos que parecem mais promissores. Já o Dijkstra explora uniformemente todas as direções, sem considerar a proximidade com o destino.

**2. O que é a Heurística Manhattan e por que ela é admissível?**  
A heurística Manhattan calcula a soma das distâncias horizontais e verticais entre dois pontos (sem diagonais). É admissível porque nunca superestima o custo real — a distância mínima sem obstáculos nunca será maior que o caminho real necessário para chegar ao destino.


## Parte C — Árvores Binárias e Percursos (DFS)

Organizando produtos em uma árvore binária de busca (BST) por preço.


In [3]:
class Node:
    def __init__(self, valor):
        self.val = valor
        self.left = None
        self.right = None

def insert(root, valor):
    if root is None:
        return Node(valor)
    if valor < root.val:
        root.left = insert(root.left, valor)
    else:
        root.right = insert(root.right, valor)
    return root

def in_order(root):
    resultado = []
    if root:
        resultado.extend(in_order(root.left))
        resultado.append(root.val)
        resultado.extend(in_order(root.right))
    return resultado

def pre_order(root):
    resultado = []
    if root:
        resultado.append(root.val)
        resultado.extend(pre_order(root.left))
        resultado.extend(pre_order(root.right))
    return resultado

def post_order(root):
    resultado = []
    if root:
        resultado.extend(post_order(root.left))
        resultado.extend(post_order(root.right))
        resultado.append(root.val)
    return resultado

precos = [50, 30, 70, 20, 40, 60, 80, 35, 45]
raiz = None
for p in precos:
    raiz = insert(raiz, p)

print("\n--- Parte C: Percursos em BST ---")
print("Em-Ordem:", in_order(raiz))
print("Pré-Ordem:", pre_order(raiz))
print("Pós-Ordem:", post_order(raiz))


--- Parte C: Percursos em BST ---
Em-Ordem: [20, 30, 35, 40, 45, 50, 60, 70, 80]
Pré-Ordem: [50, 30, 20, 40, 35, 45, 70, 60, 80]
Pós-Ordem: [20, 35, 45, 40, 30, 60, 80, 70, 50]


### Questões Discursivas — Percursos

**1. Em que situações é mais adequado usar o percurso Em-Ordem?**  
O percurso Em-Ordem é indicado para listar dados em ordem crescente (como exibir produtos ordenados por preço).

**2. Em que casos o percurso Pré-Ordem é útil?**  
O percurso Pré-Ordem é usado para copiar ou salvar a estrutura da árvore, pois processa o nó pai antes dos filhos.

**3. Quando se deve usar o percurso Pós-Ordem?**  
O percurso Pós-Ordem é ideal para processar filhos antes do pai, sendo útil ao deletar nós ou liberar memória de uma árvore.


## Parte D — Reflexões

### Questões Discursivas — Reflexões

**1. Quando o algoritmo A* pode não ser uma boa escolha?**  
Quando a heurística for muito custosa de calcular, o A* pode acabar sendo mais lento que o Dijkstra, mesmo expandindo menos nós.

**2. Qual a diferença entre corretude e otimalidade em algoritmos de busca?**  
- **Corretude:** o algoritmo encontra um caminho válido até o objetivo.  
- **Otimalidade:** o algoritmo garante o menor custo possível.  
O DFS é correto, mas não ótimo; Dijkstra e A* (com heurística admissível) são corretos e ótimos.

**3. Dê exemplos do mundo real onde cada tipo de percurso é usado.**  
- **Em-Ordem:** listar arquivos em ordem alfabética.  
- **Pré-Ordem:** processar tags XML/HTML ou salvar uma árvore.  
- **Pós-Ordem:** deletar árvores ou avaliar expressões matemáticas.

**4. O que são heurísticas inconsistentes e quais seus efeitos?**  
Heurísticas inconsistentes podem fazer com que nós sejam reabertos várias vezes, tornando o A* mais lento. No entanto, se ainda forem admissíveis, o algoritmo continua garantindo um caminho ótimo. Se superestimarem o custo, podem quebrar a otimalidade.
